# Credit scenarios

Spread widening via **par CDS / hazard** bumps (uniform and id-specific), **`instrument_price_pct_by_attr`** for sector-style shocks on tagged instruments, and a short note on **rating migration** as a conceptual step beyond parallel spreads.

## Concept

- **Uniform widening**: `OperationSpec.curve_parallel_bp` with `CurveKind.par_cds()` shifts the par CDS quotes a **calibrated** hazard curve was built from, then refits the curve exactly to the shocked quotes on the named discount curve.
- **Sector / name differentiation**: bump one hazard id more than another, or target **`OperationSpec.instrument_price_pct_by_attr`** when instruments carry metadata (e.g. `sector=Energy`).
- **Rating migration** (conceptual): a notch downgrade implies a **new spread curve** or **higher hazard**, often larger than a parallel +X bp stress—typically modeled as a **regime change**, not a tiny bump.

We compare **survival** \(S(t)\) at selected horizons before and after shocks.

In [ ]:
import json
from pathlib import Path

from finstack_quant.scenarios import (
    CurveKind,
    OperationSpec,
    ScenarioSpec,
    apply_scenario,
    build_from_template,
    build_template_component,
    compose_scenarios,
    list_builtin_templates,
    validate_scenario_spec,
)

print(
    "Imports OK",
    callable(compose_scenarios),
    callable(apply_scenario),
    callable(build_template_component),
)

print("Templates available:", list_builtin_templates())

tmeta = build_from_template("covid_2020")
print("Sample historical template covid_2020 (credit excerpt search):")
print(tmeta.to_json()[:400], "...")


PAR_CDS = CurveKind.par_cds()

# Wire format (the one raw-JSON example in this notebook): parse the complete
# scenario payload from config. Everything below uses the typed builders.
ops_uniform = json.loads(Path("data/credit_scenarios.json").read_text())["ops_uniform"]
su = ScenarioSpec.from_json(
    json.dumps(
        {
            "id": "credit_uniform",
            "name": "Uniform +75bp",
            "description": "All buckets widen",
            "operations": ops_uniform,
        }
    )
)
print("validate uniform:", validate_scenario_spec(su))

# Sector-style: widen HY more than IG (two curve IDs). `par_cds` shocks refit the
# hazard curve on the discount curve named by the trailing `discount_curve_id`.
ss = ScenarioSpec(
    "credit_sector",
    [
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 25.0, "USD-OIS"),
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HY", 150.0, "USD-OIS"),
    ],
    "IG vs HY",
    "HY wider",
)
print("validate sector:", validate_scenario_spec(ss))

# Metadata-based price shock. The builder takes an ORDERED list of `(key, value)`
# pairs; hand-written JSON that spells `attrs` as a bare map is exactly the shape
# drift the typed path removes.
sa = ScenarioSpec(
    "energy_px",
    [OperationSpec.instrument_price_pct_by_attr([("sector", "Energy")], -2.5)],
    "Energy price",
    "-2.5% on Energy-tagged instruments",
)
print("validate attr shock:", validate_scenario_spec(sa))
print("Built three credit-related scenario specs (uniform, id-specific, attribute).")


In [ ]:
from datetime import date
import math

from _shared import instrument_envelope_json
from finstack_quant.calibration import (
    CalibrationEnvelope,
    CalibrationPlan,
    CalibrationStep,
    CdsQuote,
    RateQuote,
    calibrate,
)
from finstack_quant.core.market_data import MarketContext
from finstack_quant.valuations.instruments import price_instrument

bd = date(2025, 1, 15)
AS_OF = str(bd)

# Bespoke IG/HY pair rather than `_shared.build_demo_market`: that market carries a
# single `CORP-HAZARD` curve, and the differentiated-stress lesson below needs two
# credit curves at different quality tiers. Both hazard curves are calibrated from
# par CDS quotes, so each carries the calibration recipe a `par_cds` shock refits.
rate_quotes = [
    RateQuote.deposit("USD-OIS-DEP-3M", "USD-SOFR-OIS", "3M", 0.020),
    RateQuote.swap("USD-OIS-SWAP-1Y", "USD-SOFR-OIS", "1Y", 0.020),
    RateQuote.swap("USD-OIS-SWAP-3Y", "USD-SOFR-OIS", "3Y", 0.0205),
    RateQuote.swap("USD-OIS-SWAP-5Y", "USD-SOFR-OIS", "5Y", 0.021),
]
# curve id -> (recovery rate, {tenor: par spread in bp})
credit_quotes = {
    "CORP-IG": (0.40, {"1Y": 72.2, "3Y": 96.8, "5Y": 110.5}),
    "CORP-HY": (0.35, {"1Y": 293.4, "3Y": 336.8, "5Y": 356.3}),
}
cds_quotes = {
    curve_id: [
        CdsQuote.par_spread(f"{curve_id}-{tenor}", curve_id, "USD", "isda_na", tenor, spread_bp, recovery)
        for tenor, spread_bp in spreads.items()
    ]
    for curve_id, (recovery, spreads) in credit_quotes.items()
}
plan = CalibrationPlan(
    [CalibrationStep.discount("USD-OIS", {"currency": "USD", "base_date": AS_OF})]
    + [
        CalibrationStep.hazard(
            curve_id,
            {
                "entity": curve_id,
                "currency": "USD",
                "base_date": AS_OF,
                "discount_curve_id": "USD-OIS",
                "recovery_rate": recovery,
            },
        )
        for curve_id, (recovery, _) in credit_quotes.items()
    ],
    id="ig-hy-credit",
    quote_sets={
        "USD-OIS": [quote.id for quote in rate_quotes],
        **{curve_id: [quote.id for quote in quotes] for curve_id, quotes in cds_quotes.items()},
    },
)
envelope = CalibrationEnvelope(plan, rate_quotes + [q for quotes in cds_quotes.values() for q in quotes])
mc = calibrate(envelope).market

base_ig = mc.get_hazard("CORP-IG")
base_hy = mc.get_hazard("CORP-HY")
for t in (1.0, 3.0, 5.0):
    print(f"Base survival CORP-IG t={t}Y S={base_ig.sp(t):.6f}  HY S={base_hy.sp(t):.6f}")

# Cross-check: a 0bp par-CDS shock refits each curve to its unshocked quotes, so
# survival probabilities stay finite and match the base curve.
zero_spec = ScenarioSpec(
    "zero_credit_bump",
    [
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 0.0, "USD-OIS"),
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HY", 0.0, "USD-OIS"),
    ],
    "Zero-bump calibration check",
    "",
)
zero_market = apply_scenario(zero_spec, mc, AS_OF).market

for curve_id in ("CORP-IG", "CORP-HY"):
    zero_curve = zero_market.get_hazard(curve_id)
    survival = [zero_curve.sp(t) for t in (1.0, 3.0, 5.0)]
    assert all(math.isfinite(probability) and 0.0 <= probability <= 1.0 for probability in survival)
    base_curve = mc.get_hazard(curve_id)
    assert all(
        math.isclose(probability, base_curve.sp(t), abs_tol=1e-8)
        for probability, t in zip(survival, (1.0, 3.0, 5.0))
    )

def apply_and_print(label: str, spec: ScenarioSpec):
    result = apply_scenario(spec, mc, AS_OF)
    print(f"\n--- {label} ---")
    print(
        "  operations_applied:",
        result.report.operations_applied,
        " warnings:",
        result.report.warnings,
    )
    stressed_market = result.market
    stressed_ig = stressed_market.get_hazard("CORP-IG")
    stressed_hy = stressed_market.get_hazard("CORP-HY")
    for t in (1.0, 3.0, 5.0):
        print(
            f"  t={t}Y  IG S {base_ig.sp(t):.6f} -> {stressed_ig.sp(t):.6f}  |  HY {base_hy.sp(t):.6f} -> {stressed_hy.sp(t):.6f}"
        )

spec_uniform = ScenarioSpec(
    "u",
    [OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 80.0, "USD-OIS")],
    "IG +80bp",
    "Uniform widening on the IG curve only",
)

spec_both = ScenarioSpec(
    "b",
    [
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 30.0, "USD-OIS"),
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HY", 120.0, "USD-OIS"),
    ],
    "IG +30bp, HY +120bp",
    "Differentiated widening across quality tiers",
)

apply_and_print("IG-only +80bp par CDS shock", spec_uniform)
apply_and_print("Sector-style IG +30bp, HY +120bp", spec_both)

# Reprice a single-name CDS off CORP-IG under base vs IG-only stressed market.
# Written out here rather than taken from `_shared.instrument_fixtures.cds` because
# that fixture references `CORP-HAZARD`; this trade must reference `CORP-IG`.
cds_ig = instrument_envelope_json(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-ON-IG",
            "notional": {"amount": "1000000", "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium_leg": {
                "start": AS_OF,
                "end": "2030-01-15",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "short_front",
                "business_day_convention": "following",
                "calendar_id": None,
                "day_count": "act_360",
                "coupon_bp": "120",
                "date_regime": "bespoke",  # Bespoke January maturity, not IMM dates.
                "discount_curve_id": "USD-OIS",
            },
            "protection_leg": {
                "credit_curve_id": "CORP-IG",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "attributes": {},
        },
    }
)

def cds_pv_usd(market: MarketContext) -> float:
    rj = price_instrument(cds_ig, market, AS_OF, model="hazard_rate")
    return rj.price

ru = apply_scenario(spec_uniform, mc, AS_OF)
pv_base = cds_pv_usd(mc)
pv_stress = cds_pv_usd(ru.market)
print(
    f"\nCDS (CORP-IG) PV — base vs IG-only +80bp market: {pv_base:,.2f} -> {pv_stress:,.2f} (Δ {pv_stress - pv_base:+,.2f} USD)"
)

print("\nRating migration (conceptual): a multi-notch move usually implies re-calibrating to a new curve level/shape, not a single parallel bump.")


## Takeaways

- **`CurveKind.par_cds()`** shocks are the primary way to stress **credit curves** tied to **`HazardCurve`** ids; pass **`discount_curve_id`** to name the discount curve the refit uses.
- A `par_cds` shock is an **exact par-CDS refit**: the engine shifts the curve's calibration quotes and re-solves every hazard segment so the shocked CDS reprice to par. It needs a hazard curve built by **calibration** (which carries the replay recipe); a hand-built `HazardCurve` is rejected with a `ValueError`.
- **Uniform vs differentiated** credit stress is expressed by **one shared bump** or **different `curve_id` targets** (and magnitudes) in the same operation list.
- **`OperationSpec.instrument_price_pct_by_attr`** is useful when positions carry **metadata** you want to shock as a group; it takes an **ordered list of `(key, value)` pairs**, which is why the typed builder is safer than hand-writing the `attrs` object.
- A **0bp par-CDS shock** is a cheap apply-path cross-check: the refit reproduces the base curve, so survival probabilities stay finite and unchanged.
- **Survival** drops when spreads widen—use **`result.market.get_hazard`** to inspect \(S(t)\).